# GM Tube Characterisation

Complete characterisation of Geiger-Müller detector tubes using the Arduino-based GM counter.

| Step                  | Measurement           | Automation                    | Duration (default)      |
| --------------------- | --------------------- | ----------------------------- | ----------------------- |
| 0.5 Rate Scout        | Rate at V_OPERATE     | Fully automated               | ~15 s                   |
| 1. Geiger Plateau     | Count rate vs HV      | Fully automated               | ~10 min (20 pts × 30 s) |
| 2. Dead Time          | Inter-event timing    | Fully automated               | ~2–5 min                |
| 3. Distance Law       | Rate vs distance      | Semi-auto (user moves source) | ~5 min                  |
| 4. Background         | No-source count rate  | Fully automated               | ~5 min                  |
| 5. Poisson Statistics | Count distribution    | Fully automated               | ~20 min (100 × 10 s)    |

**Hardware:** Arduino GM counter connected via USB. See `gmcounter/infrastructure/devices/gm_counter.py`.

**Mock mode:** Set `PORT = "mock"` to run without hardware using the PTY-based simulator.

**Source guidance:** Use a **Cs-137** source in preference to Co-60 — its 662 keV γ and energetic β
give higher detection efficiency in a typical GM tube, which directly translates to more counts per
point and better statistical precision on the plateau. Aim for 30–200 cps at the operating voltage
(adjust distance accordingly). If the source is so active that dead-time losses exceed ~5 %, move
it farther away.

## Configuration — edit this cell


In [ ]:
from serial.tools import list_ports

print("Available serial ports:")
for port in list_ports.comports():
    print(f"  {port.device}")

In [ ]:
# ── Device ─────────────────────────────────────────────────────────────────
PORT = "/dev/cu.usbmodem1101"  # e.g. "/dev/tty.usbmodem14101" or "mock"
BAUDRATE = 1_000_000

# ── Tube under test ────────────────────────────────────────────────────────
TUBE_ID = "E94155"  # E-prefix 5-digit code stamped on the tube
TUBE_SIZE = "45mm"  # "15mm" or "45mm"

# ── Radioactive source ─────────────────────────────────────────────────────
SOURCE_ID = "E54024"  # E-prefix code, or "BACKGROUND" for no source

# ── Measurement durations ──────────────────────────────────────────────────
# Counting-time keys:  1→1 s  2→10 s  3→60 s  4→100 s  5→300 s
CT_PLATEAU = 2  # per plateau point  (key=3 → 60 s)
CT_DISTANCE = 4  # per distance point (key=4 → 100 s)
CT_POISSON = 3  # per Poisson sample (key=3 → 60 s)
CT_BACKGROUND = 5  # background run     (key=5 → 300 s)
N_POISSON = 100  # repeated measurements for Poisson test
N_TIMING = 8_000  # inter-event samples for dead-time analysis

# ── Plateau sweep ──────────────────────────────────────────────────────────
V_START, V_STOP, V_STEP = 300, 700, 20  # volts
V_STABILISE_S = 3.0  # wait after voltage change

# ── Operating voltage (used for all non-plateau measurements) ──────────────
V_OPERATE = 500  # set after plateau is measured

# ── Mock rate (ignored when using real hardware) ───────────────────────────
MOCK_MAX_TICK_S = 0.015  # upper bound of inter-event time → ~67 cps mean

## Imports and helper functions


In [ ]:
import csv
import math
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Add project root to path so gmcounter package is importable
PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── Output directory ───────────────────────────────────────────────────────
# All raw CSVs land here; directory is created automatically on first save.
OUT_DIR = Path(f"results/{TUBE_ID}")

from gmcounter.infrastructure.devices.gm_counter import GMCounterAdapter
from gmcounter.infrastructure.packet_parser import PacketParser

CT_MAP = {0: 0, 1: 1, 2: 10, 3: 60, 4: 100, 5: 300}  # key → seconds

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.4})

print("Imports OK")

In [ ]:
# -- Connection helpers ---------------------------------------------------
def start_mock_server(max_tick_s=0.015):
    from gmcounter.infrastructure.mocks.mock_gm_counter import run_pty_server, PORT_FILE
    import threading

    port_file = Path(PORT_FILE)
    if port_file.exists():
        port_file.unlink()
    t = threading.Thread(
        target=run_pty_server, kwargs={"max_tick": max_tick_s}, daemon=True
    )
    t.start()
    deadline = time.time() + 5.0
    while not port_file.exists() and time.time() < deadline:
        time.sleep(0.05)
    if not port_file.exists():
        raise RuntimeError("Mock server did not start in time")
    port = port_file.read_text().strip()
    print(f"  Mock PTY server started on {port}")
    return port


def connect(port, baudrate=BAUDRATE):
    if port == "mock":
        port = start_mock_server(max_tick_s=MOCK_MAX_TICK_S)
    gm = GMCounterAdapter(port=port, baudrate=baudrate)
    print(f"  Connected to {port}")
    return gm


# -- Save helper -----------------------------------------------------------


def save_raw(measurement_tag, columns, rows, extra_meta=None):
    """Write raw measurement data as a self-describing CSV.

    Filename: {OUT_DIR}/YYYY-MM-DD_HHmm_{measurement_tag}_{TUBE_ID}_{SOURCE_ID}.csv
    Header:   # key: value  comment lines, then a standard CSV body.
    """
    ts = datetime.now()
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    fname = f"{ts:%Y-%m-%d}_{ts:%H%M}_{measurement_tag}_{TUBE_ID}_{SOURCE_ID}.csv"
    path = OUT_DIR / fname

    meta = {
        "measurement": measurement_tag,
        "date": ts.strftime("%Y-%m-%d"),
        "time": ts.strftime("%H:%M:%S"),
        "tube_id": TUBE_ID,
        "tube_size": TUBE_SIZE,
        "source_id": SOURCE_ID,
        "operating_voltage_V": V_OPERATE,
    }
    if extra_meta:
        meta.update(extra_meta)

    with open(path, "w", newline="", encoding="utf-8") as fh:
        for k, v in meta.items():
            fh.write(f"# {k}: {v}\n")
        w = csv.writer(fh)
        w.writerow(columns)
        w.writerows(rows)

    print(f"  -> {path}")
    return path


# -- Measurement helpers ---------------------------------------------------


def measure_timed(gm, ct_key, verbose=True, label=""):
    ct_s = CT_MAP[ct_key]
    if ct_s == 0:
        raise ValueError("ct_key must be 1-5 for timed measurements")
    gm.set_counting(False)
    gm.set_stream(0)
    gm.set_repeat(False)
    gm.set_counting_time(ct_key)
    gm.flush_input_buffer()
    gm.set_counting(True)
    t0 = time.time()
    while time.time() - t0 < ct_s:
        elapsed = time.time() - t0
        if verbose:
            print(f"\r  {label}  {elapsed:4.0f}/{ct_s} s", end="", flush=True)
        time.sleep(0.5)
    gm.set_counting(False)
    time.sleep(1.5)
    if verbose:
        print()
    data = gm.get_data(request=True)
    while data["error"] == 1:
        # print("  Error reading data, retrying...")
        time.sleep(1.0)
        data = gm.get_data(request=True)
    count = data["last_count"]
    return count, count / ct_s


def read_timing_stream(gm, n_events=5000, max_seconds=180, ticks_per_us=48):
    parser = PacketParser(ticks_per_us=float(ticks_per_us))
    gm.set_counting(False)
    gm.set_counting_time(0)
    gm.set_stream(1)
    gm.flush_input_buffer()
    gm.set_counting(True)
    intervals = []
    deadline = time.time() + max_seconds
    last_print = time.time()
    while len(intervals) < n_events and time.time() < deadline:
        raw = gm.read_fast(max_bytes=8192, timeout_ms=50)
        if raw:
            points = parser.feed(raw)
            intervals.extend(v for _, v in points)
        if time.time() - last_print > 1.0:
            print(f"\r  {len(intervals)}/{n_events} events", end="", flush=True)
            last_print = time.time()
    gm.set_counting(False)
    gm.set_stream(0)
    gm.flush_input_buffer()
    print(f"\n  Collected {len(intervals)} events")
    return np.array(intervals[:n_events])


print("Helpers defined")

---

## 0 · Connect and identify


In [ ]:
gm = connect(PORT)

info = gm.get_information(use_cache=False)
print(f"  Firmware / device : {info.get('version', 'n/a')}")
print(f"  Serial / openbis  : {info.get('openbis', 'n/a')}")
print(f"  Copyright         : {info.get('copyright', 'n/a')}")
print()
print(f"  Tube under test   : {TUBE_ID}  ({TUBE_SIZE})")

# Set operating voltage
gm.set_voltage(V_OPERATE)
time.sleep(V_STABILISE_S)
data = gm.get_data(request=True)
print(f"  Readback voltage  : {data['voltage']} V")

---

## 0.5 · Rate Scout — choose integration times

A short count at the operating voltage tells you the actual count rate before committing to a full sweep.
From this you can pick integration times that guarantee a target relative statistical uncertainty.

**Rule of thumb:** relative uncertainty ≈ 1/√N, so for 5 % uncertainty you need N ≥ 400 counts per point.

In [ ]:
# ── Rate scout: one short count to calibrate integration times ─────────────
SCOUT_CT_KEY = 2  # 10 s — fast enough to not matter, long enough to be stable

gm.set_voltage(V_OPERATE)
time.sleep(V_STABILISE_S)

scout_count, scout_rate = measure_timed(gm, SCOUT_CT_KEY, verbose=True, label="Scout")
print(f"  Rate at {V_OPERATE} V : {scout_rate:.2f} cps  ({scout_count} counts in {CT_MAP[SCOUT_CT_KEY]} s)")
print()

TARGET_UNC = 0.05  # 5 % relative uncertainty target
n_target = 1.0 / TARGET_UNC**2  # N counts needed for that uncertainty

print(f"Integration-time recommendations for {TARGET_UNC*100:.0f} % relative uncertainty")
print(f"  (need N \u2265 {n_target:.0f} counts per point at {scout_rate:.2f} cps)\n")
print(f"  {'Key':>4}  {'Time (s)':>9}  {'Expected N':>12}  {'Rel. unc.':>10}  Suitable?")
print(f"  {'-'*55}")
for key in [1, 2, 3, 4, 5]:
    t_s = CT_MAP[key]
    n_exp = scout_rate * t_s
    rel_unc = 1.0 / math.sqrt(max(n_exp, 1))
    ok = "YES" if rel_unc <= TARGET_UNC else "no"
    print(f"  {key:>4}  {t_s:>9}  {n_exp:>12.0f}  {rel_unc*100:>9.1f} %  {ok}")

if scout_rate > 0:
    t_min = n_target / scout_rate
    print(f"\n  Minimum integration time for {TARGET_UNC*100:.0f} % uncertainty: {t_min:.0f} s")
    print(f"  \u2192 update CT_PLATEAU in the config cell if needed before running section 1.")

---

## 1 · Geiger Plateau _(automated)_

Sweep the high voltage from `V_START` to `V_STOP` in `V_STEP` increments and measure count
rate at each point. Identifies the plateau region, plateau slope, and recommended operating point.

**Duration:** `N_pts × (CT_PLATEAU_s + V_STABILISE_S)` seconds.


In [ ]:
plateau_voltages = list(range(V_START, V_STOP + 1, V_STEP))
plateau_counts = []
plateau_rates = []
ct_s_plateau = CT_MAP[CT_PLATEAU]

total = len(plateau_voltages)
est_minutes = total * (ct_s_plateau + V_STABILISE_S) / 60
print(
    f"Plateau sweep: {total} points x ({ct_s_plateau} s + {V_STABILISE_S:.0f} s stabilise) approx {est_minutes:.1f} min"
)
print()

for i, v in enumerate(plateau_voltages):
    gm.flush_input_buffer()
    resp = gm.set_voltage(v, True)
    while resp is False:
        # print(f"  Voltage set failed for {v} V, retrying...")
        time.sleep(1.0)
        resp = gm.set_voltage(v, True)

    count, rate = measure_timed(
        gm, CT_PLATEAU, verbose=True, label=f"[{i + 1:2d}/{total}]  {v} V"
    )
    # for i in range(5):
    #     time.sleep(1.0)
    #     data = gm.get_data(request=True)
    #     print(data)
    plateau_counts.append(count)
    plateau_rates.append(rate)
    print(f"           {v} V -> {count} counts  ({rate:.2f} cps)")

gm.set_voltage(V_OPERATE)
print(f"\nDone. Operating voltage restored to {V_OPERATE} V.")

save_raw(
    "plateau",
    ["voltage_V", "count", "counting_time_s", "rate_cps", "rate_err_cps"],
    [
        [v, c, ct_s_plateau, f"{r:.6f}", f"{math.sqrt(max(c, 1)) / ct_s_plateau:.6f}"]
        for v, c, r in zip(plateau_voltages, plateau_counts, plateau_rates)
    ],
    extra_meta={
        "v_start": V_START,
        "v_stop": V_STOP,
        "v_step_V": V_STEP,
        "stabilise_s": V_STABILISE_S,
    },
)

In [ ]:
# ── Plateau analysis ───────────────────────────────────────────────────────
v_arr = np.array(plateau_voltages)
r_arr = np.array(plateau_rates)
r_err = np.sqrt(np.maximum(plateau_counts, 1)) / ct_s_plateau  # Poisson √N

# Threshold voltage: first point > 5 % of max rate
max_rate = r_arr.max()
threshold_idx = np.argmax(r_arr > 0.05 * max_rate)
v_threshold = v_arr[threshold_idx]

# Plateau region: voltages where rate is within 20 % of the median plateau rate
# (use the middle 50 % of points as initial plateau estimate)
mid = r_arr[len(r_arr) // 4 : 3 * len(r_arr) // 4]
r_mid = np.median(mid)
plat_mask = (r_arr >= 0.80 * r_mid) & (r_arr <= 1.30 * r_mid)
v_plat = v_arr[plat_mask]

# Plateau slope (linear fit over plateau region, %/100V)
if len(v_plat) >= 3:
    slope_fit = np.polyfit(v_plat, r_arr[plat_mask], 1)
    plateau_slope_pct = slope_fit[0] / r_mid * 100 * 100  # %/100V
    v_recommend = v_plat[len(v_plat) // 3]  # lower third of plateau
else:
    plateau_slope_pct = float("nan")
    v_recommend = V_OPERATE

# ── Plot ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: rate vs voltage
ax = axes[0]
ax.errorbar(
    v_arr,
    r_arr,
    yerr=r_err,
    fmt="o-",
    ms=5,
    capsize=4,
    color="steelblue",
    label="Count rate",
)
if len(v_plat) >= 2:
    ax.axvspan(v_plat[0], v_plat[-1], alpha=0.15, color="green", label="Plateau region")
ax.axvline(
    v_threshold, color="orange", ls="--", lw=1.5, label=f"Threshold ≈ {v_threshold} V"
)
if not math.isnan(plateau_slope_pct):
    ax.axvline(
        v_recommend,
        color="red",
        ls=":",
        lw=2,
        label=f"Recommended op. pt. {v_recommend} V",
    )
ax.set_xlabel("High Voltage (V)")
ax.set_ylabel("Count rate (cps)")
ax.set_title(f"Geiger Plateau — {TUBE_ID} ({TUBE_SIZE})")
ax.legend(fontsize=9)

# Right: derivative (sensitivity to voltage, plateau flatness)
ax2 = axes[1]
if len(v_arr) > 2:
    dr = np.gradient(r_arr, v_arr)
    ax2.plot(v_arr, dr, "s-", color="tomato", ms=4)
    ax2.axhline(0, color="gray", lw=0.8)
    ax2.set_xlabel("High Voltage (V)")
    ax2.set_ylabel("d(rate)/dV  (cps/V)")
    ax2.set_title("Plateau sensitivity")

plt.suptitle(
    f"GM Tube Characterisation — Geiger Plateau  [{datetime.now():%Y-%m-%d}]",
    fontsize=11,
    y=1.01,
)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/voltage_plateau-{TUBE_ID}.pdf", dpi=300, bbox_inches="tight")
plt.show()

print(f"Threshold voltage      : {v_threshold} V")
print(
    f"Plateau region         : {v_plat[0] if len(v_plat) else 'n/a'} – "
    f"{v_plat[-1] if len(v_plat) else 'n/a'} V"
)
print(f"Plateau slope          : {plateau_slope_pct:.2f} %/100V  (ideal < 10 %)")
print(f"Max count rate         : {max_rate:.2f} cps at {v_arr[np.argmax(r_arr)]} V")
print(f"Recommended op. point  : {v_recommend} V")

---

## 2 · Dead Time from Timing Distribution _(automated)_

Reads the raw inter-event stream and fits a **shifted exponential** to the
distribution. For a non-paralyzable (Type I) detector the inter-arrival PDF is:

$$f(t) = \lambda \, e^{-\lambda(t - \tau)} \quad t \geq \tau$$

where $\tau$ is the dead time and $\lambda$ is the _true_ event rate.

**Two estimators are shown:**

- **MLE minimum-statistic:** $\hat{\tau} = \min(t_i)$, bias-corrected by $-1/(n\hat{\lambda})$.
- **Histogram onset:** leftmost bin that exceeds 5 % of the histogram peak.

A reduced $\chi^2$ assesses which model fits better.


In [ ]:
gm.set_voltage(V_OPERATE)
time.sleep(V_STABILISE_S)

print(f"Collecting {N_TIMING} inter-event samples at {V_OPERATE} V ...")
intervals_us = read_timing_stream(gm, n_events=N_TIMING, max_seconds=300)

save_raw(
    "deadtime_stream",
    ["index", "interval_us"],
    [[i, f"{v:.4f}"] for i, v in enumerate(intervals_us)],
    extra_meta={"n_events": len(intervals_us), "voltage_V": V_OPERATE},
)

In [ ]:
# ── Dead-time analysis ──────────────────────────────────────────────────────
def mle_shifted_exp(t):
    """MLE for shifted exponential. Returns (tau_mle, lambda)."""
    tau = t.min()
    lam = 1.0 / (t.mean() - tau)
    return tau, lam


def bias_correct_tau(tau, lam, n):
    return tau - 1.0 / (n * lam)


def histogram_onset(t, bins=300, threshold_frac=0.05):
    counts, edges = np.histogram(t, bins=bins)
    peak = counts.max()
    for i, c in enumerate(counts):
        if c >= threshold_frac * peak:
            return edges[i]
    return edges[0]


def reduced_chi2(t, tau, lam, bins=80):
    """Reduced chi² of shifted-exp fit vs histogram (bins with expected >= 5)."""
    counts, edges = np.histogram(t, bins=bins)
    n = len(t)
    chi2, dof = 0.0, 0
    for i, count in enumerate(counts):
        lo, hi = edges[i], edges[i + 1]
        if hi <= tau:
            continue
        lo_eff = max(lo, tau)
        expected = n * (np.exp(-lam * (lo_eff - tau)) - np.exp(-lam * (hi - tau)))
        if expected >= 5:
            chi2 += (counts[i] - expected) ** 2 / expected
            dof += 1
    return chi2 / max(dof - 2, 1)


def deadtime_correction(measured_cps, tau_us):
    """Non-paralyzable correction: n = m / (1 - m*tau)."""
    tau_s = tau_us * 1e-6
    if measured_cps * tau_s >= 1.0:
        return None
    true_cps = measured_cps / (1.0 - measured_cps * tau_s)
    return true_cps, (true_cps - measured_cps) / true_cps


t = intervals_us[intervals_us > 0]
tau_mle, lam = mle_shifted_exp(t)
tau_bc = bias_correct_tau(tau_mle, lam, len(t))
tau_hist = histogram_onset(t)
chi2r = reduced_chi2(t, tau_mle, lam)
measured_cps = 1e6 / t.mean()
cv = t.std() / t.mean()

print(f"Events analysed       : {len(t):,}")
print(f"Observed mean rate    : {measured_cps:.2f} cps")
print(f"Coeff. of variation   : {cv:.4f}  (1.0 = ideal Poisson)")
print()
print(f"── Dead-time estimates ─────────────────────────")
print(f"  τ (MLE, biased)     : {tau_mle:.2f} µs")
print(f"  τ (bias-corrected)  : {tau_bc:.2f} µs  ← use this")
print(f"  τ (histogram onset) : {tau_hist:.2f} µs")
print(
    f"  Reduced χ²          : {chi2r:.3f}  ({'good' if chi2r < 2 else 'moderate' if chi2r < 5 else 'poor'} fit)"
)
print(f"  True rate (MLE)     : {lam * 1e6:.2f} cps")

corr = deadtime_correction(measured_cps, tau_bc)
if corr:
    true_cps, loss = corr
    print()
    print(f"── Rate correction (non-paralyzable) ───────────")
    print(f"  Measured rate       : {measured_cps:.2f} cps")
    print(f"  Corrected true rate : {true_cps:.2f} cps")
    print(f"  Dead-time loss      : {loss * 100:.2f} %")

In [ ]:
# ── Dead-time plot ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

t_fit = np.linspace(tau_mle, np.percentile(t, 99), 600)
pdf_fit = lam * np.exp(-lam * (t_fit - tau_mle))

for ax, log_scale in zip(axes, [False, True]):
    ax.hist(
        t,
        bins=120,
        density=True,
        alpha=0.6,
        color="steelblue",
        label="Inter-event data",
    )
    ax.plot(
        t_fit,
        pdf_fit,
        "r-",
        lw=2,
        label=f"Shifted-exp fit\nτ={tau_mle:.1f} µs, λ={lam:.5g}/µs",
    )
    ax.axvline(
        tau_mle, color="tomato", ls="--", lw=1.5, label=f"τ MLE = {tau_mle:.1f} µs"
    )
    ax.axvline(
        tau_bc, color="darkorange", ls=":", lw=2, label=f"τ bias-corr = {tau_bc:.1f} µs"
    )
    ax.axvline(
        tau_hist, color="purple", ls="-.", lw=1.5, label=f"τ hist = {tau_hist:.1f} µs"
    )
    ax.set_xlabel("Inter-event time (µs)")
    ax.set_ylabel("Probability density" + (" [log]" if log_scale else ""))
    ax.set_xlim(left=0, right=np.percentile(t, 99.5))
    if log_scale:
        ax.set_yscale("log")
    ax.legend(fontsize=8)
    ax.set_title("Inter-event distribution" + (" (log scale)" if log_scale else ""))

plt.suptitle(
    f"Dead-Time Analysis — {TUBE_ID}  n={len(t):,}  [{datetime.now():%Y-%m-%d}]",
    fontsize=11,
    y=1.01,
)
plt.tight_layout()
plt.savefig(f"deadtime_{TUBE_ID}.png", dpi=150, bbox_inches="tight")
plt.show()

---

## 3 · Distance Dependence _(semi-automated)_

Verifies the **inverse-square law** $\dot{N} \propto 1/r^2$. Place the source at each
distance in `DISTANCES_CM`, press **Run** on the measurement cell, then **Run** on the plot cell.

**Tube comparison:** repeat this section with each tube at the same distances and compare
efficiencies from the fit intercepts.


In [ ]:
# ── Edit distances here ────────────────────────────────────────────────────
DISTANCES_CM = [2, 4, 6, 8, 10, 15, 20, 30]  # cm, source to tube face
# Run this cell ONCE; then run the measurement cell below for each distance,
# or leave DISTANCES_CM as-is and run the loop cell to measure all at once.

distance_results = []  # will hold (r_cm, count, rate_cps) tuples
print(f"Ready. {len(DISTANCES_CM)} distances queued: {DISTANCES_CM} cm")
print(f"Measurement time per point: {CT_MAP[CT_DISTANCE]} s")

In [ ]:
gm.set_voltage(V_OPERATE)
time.sleep(V_STABILISE_S)
distance_results.clear()

for r in DISTANCES_CM:
    input(f"  Place source at {r} cm and press Enter ...")
    print(f"  Measuring {CT_MAP[CT_DISTANCE]} s at {r} cm ...")
    count, rate = measure_timed(gm, CT_DISTANCE, verbose=True, label=f"r={r} cm")
    distance_results.append((r, count, rate))
    print(f"  {r} cm -> {count} counts  ({rate:.2f} cps)")

print("\nAll distance measurements done.")

save_raw(
    "distance",
    ["distance_cm", "count", "counting_time_s", "rate_cps", "rate_err_cps"],
    [
        [
            r,
            c,
            CT_MAP[CT_DISTANCE],
            f"{rt:.6f}",
            f"{math.sqrt(max(c, 1)) / CT_MAP[CT_DISTANCE]:.6f}",
        ]
        for r, c, rt in distance_results
    ],
)

In [ ]:
# ── 1/r² fit and plot ──────────────────────────────────────────────────────
if not distance_results:
    print("No data yet — run the measurement cell first.")
else:
    r_cm = np.array([x[0] for x in distance_results])
    counts = np.array([x[1] for x in distance_results])
    rates = np.array([x[2] for x in distance_results])
    r_err = np.sqrt(np.maximum(counts, 1)) / CT_MAP[CT_DISTANCE]

    inv_r2 = 1.0 / r_cm**2

    # Subtract background if measured in Section 4, else zero
    bg_rate = globals().get("background_rate", 0.0)
    net_rates = rates - bg_rate

    # Linear fit: rate = A/r²  (weighted by Poisson uncertainty)
    weights = 1.0 / np.maximum(r_err**2, 1e-9)
    coeffs = np.polyfit(inv_r2, net_rates, 1, w=weights)
    fit_line = np.poly1d(coeffs)
    r2_range = np.linspace(inv_r2.min(), inv_r2.max(), 200)

    residuals = net_rates - fit_line(inv_r2)
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((net_rates - net_rates.mean()) ** 2)
    r_squared = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    ax.errorbar(
        inv_r2,
        net_rates,
        yerr=r_err,
        fmt="o",
        ms=6,
        capsize=4,
        color="steelblue",
        label="Data",
    )
    ax.plot(
        r2_range,
        fit_line(r2_range),
        "r-",
        lw=2,
        label=f"Linear fit  R²={r_squared:.4f}",
    )
    ax.set_xlabel("1/r²  (cm⁻²)")
    ax.set_ylabel("Net count rate (cps)")
    ax.set_title(f"Distance Law — {TUBE_ID}")
    ax.legend()

    ax2 = axes[1]
    ax2.errorbar(r_cm, rates, yerr=r_err, fmt="o-", ms=6, capsize=4, color="steelblue")
    r_plot = np.linspace(r_cm.min(), r_cm.max(), 200)
    ax2.plot(
        r_plot,
        fit_line(1 / r_plot**2) + bg_rate,
        "r-",
        lw=2,
        label="1/r² fit + background",
    )
    ax2.set_xlabel("Distance r (cm)")
    ax2.set_ylabel("Count rate (cps)")
    ax2.set_title("Count rate vs distance")
    ax2.legend()

    plt.suptitle(
        f"Distance Law — {TUBE_ID} ({TUBE_SIZE})  [{datetime.now():%Y-%m-%d}]",
        fontsize=11,
        y=1.01,
    )
    plt.tight_layout()
    plt.savefig(f"distance_{TUBE_ID}.png", dpi=150, bbox_inches="tight")
    plt.show()

    print(f"Inverse-square fit:  rate = {coeffs[0]:.4g} / r²  +  {coeffs[1]:.4g}")
    print(f"Fit quality R²     : {r_squared:.5f}")
    print(
        f"Solid-angle factor : {coeffs[0]:.4g}  cps·cm²  (proportional to detector efficiency × source activity)"
    )

---

## 4 · Background Measurement _(automated)_

Measure the background count rate with no radioactive source present.


In [ ]:
gm.set_voltage(V_OPERATE)
time.sleep(V_STABILISE_S)

print(f"Measuring background for {CT_MAP[CT_BACKGROUND]} s ...")
bg_count, background_rate = measure_timed(gm, CT_BACKGROUND, verbose=True, label="BG")

bg_rate_err = math.sqrt(max(bg_count, 1)) / CT_MAP[CT_BACKGROUND]
print(f"Background count  : {bg_count}")
print(f"Background rate   : {background_rate:.3f} +/- {bg_rate_err:.3f}  cps")
print(f"                  : {background_rate * 60:.2f} cpm")

save_raw(
    "background",
    ["count", "counting_time_s", "rate_cps", "rate_err_cps"],
    [[bg_count, CT_MAP[CT_BACKGROUND], f"{background_rate:.6f}", f"{bg_rate_err:.6f}"]],
)

---

## 5 · Poisson Statistics Test _(automated)_

Collects `N_POISSON` repeated measurements and tests whether the count distribution
follows the expected Poisson distribution.

**Metrics:**

- **Fano factor** $F = \sigma^2/\bar{n}$ — should be ≈ 1 for Poisson.
- **Pearson χ² test** — tests goodness-of-fit to Poisson.
- **Variance/mean** (same as Fano but phrased differently).
- **Index of dispersion** test (Bartlett).


In [ ]:
gm.set_voltage(V_OPERATE)
time.sleep(V_STABILISE_S)

ct_s_pois = CT_MAP[CT_POISSON]
est_min = N_POISSON * (ct_s_pois + 1.5) / 60
print(f"Collecting {N_POISSON} x {ct_s_pois} s measurements  (~{est_min:.1f} min) ...")

poisson_counts = []
for i in range(N_POISSON):
    count, _ = measure_timed(gm, CT_POISSON, verbose=False)
    poisson_counts.append(count)
    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/{N_POISSON}  running mean = {np.mean(poisson_counts):.1f}")

print("Done.")

save_raw(
    "poisson",
    ["sample", "count"],
    [[i, c] for i, c in enumerate(poisson_counts)],
    extra_meta={"counting_time_s": ct_s_pois, "n_samples": N_POISSON},
)

In [ ]:
# ── Poisson analysis ────────────────────────────────────────────────────────
c = np.array(poisson_counts)
mu = c.mean()
var = c.var(ddof=1)
fano = var / mu

# Pearson chi-squared goodness-of-fit
lo, hi = max(0, int(mu - 4 * math.sqrt(mu))), int(mu + 6 * math.sqrt(mu)) + 1
observed_bins = np.bincount(c.clip(lo, hi) - lo, minlength=hi - lo + 1)
k_vals = np.arange(lo, hi + 1)
expected_bins = len(c) * stats.poisson.pmf(k_vals, mu)

# Pool small bins (expected < 5) for valid chi-squared
mask = expected_bins >= 5
if mask.sum() >= 3:
    chi2_stat = np.sum(
        (observed_bins[mask] - expected_bins[mask]) ** 2 / expected_bins[mask]
    )
    dof = mask.sum() - 1
    p_value = 1.0 - stats.chi2.cdf(chi2_stat, dof)
else:
    chi2_stat = p_value = float("nan")
    dof = 0

# Kolmogorov-Smirnov (empirical CDF vs Poisson CDF)
ks_stat, ks_p = stats.kstest(c, lambda x: stats.poisson.cdf(x, mu))

print(f"N samples          : {len(c)}")
print(f"Mean count µ̂      : {mu:.4f}  ({mu / ct_s_pois:.3f} cps)")
print(f"Variance σ²        : {var:.4f}")
print(f"Fano factor F=σ²/µ : {fano:.4f}  (1.000 = ideal Poisson)")
print(f"Pearson χ²         : {chi2_stat:.3f}  df={dof}  p={p_value:.4f}")
print(f"KS test            : D={ks_stat:.4f}  p={ks_p:.4f}")
print()
verdict = (
    "consistent with Poisson"
    if (not math.isnan(p_value) and p_value > 0.05)
    else "deviation from Poisson"
)
print(f"Verdict            : {verdict}")

In [ ]:
# ── Poisson plot ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
bins_plot = np.arange(lo - 0.5, hi + 1.5)
ax.hist(c, bins=bins_plot, density=True, alpha=0.7, color="steelblue", label="Measured")
k_fine = np.arange(lo, hi + 1)
ax.stem(
    k_fine,
    stats.poisson.pmf(k_fine, mu),
    linefmt="r-",
    markerfmt="ro",
    basefmt=" ",
    label=f"Poisson(µ={mu:.2f})",
)
ax.set_xlabel(f"Counts per {ct_s_pois} s interval")
ax.set_ylabel("Relative frequency")
ax.set_title(f"Count Distribution — {TUBE_ID}")
ax.legend()

# QQ-plot against Poisson
ax2 = axes[1]
c_sorted = np.sort(c)
quantiles = (np.arange(1, len(c) + 1) - 0.5) / len(c)
theory_q = stats.poisson.ppf(quantiles, mu)
ax2.plot(theory_q, c_sorted, "o", ms=3, alpha=0.5, color="steelblue", label="Data")
lims = [
    min(theory_q.min(), c_sorted.min()) - 1,
    max(theory_q.max(), c_sorted.max()) + 1,
]
ax2.plot(lims, lims, "r--", lw=1.5, label="Ideal Poisson")
ax2.set_xlabel("Theoretical quantile (Poisson)")
ax2.set_ylabel("Observed quantile")
ax2.set_title("QQ Plot (Poisson)")
ax2.legend()

plt.suptitle(
    f"Poisson Statistics — {TUBE_ID}  F={fano:.3f}  [{datetime.now():%Y-%m-%d}]",
    fontsize=11,
    y=1.01,
)
plt.tight_layout()
plt.savefig(f"poisson_{TUBE_ID}.png", dpi=150, bbox_inches="tight")
plt.show()

---

## 6 · Multi-Tube Comparison

After running sections 1–5 for each tube, collect the key results in a comparison table.

Typical differences between **15 mm** and **45 mm** tubes:

- Larger tube → higher count rate (more sensitive area)
- May differ in threshold voltage and plateau slope
- Dead times should be similar (dominated by electronics, ~100–400 µs)


In [ ]:
# ── Accumulate results across runs ─────────────────────────────────────────
# Run this cell once per tube after completing sections 1-5 for that tube.
# Results are stored in `tube_summary` (a list of dicts).

if "tube_summary" not in dir():
    tube_summary = []

# Collect values from current tube's measurements (uses variables set above)
entry = {
    "tube_id": TUBE_ID,
    "size": TUBE_SIZE,
    "v_threshold": v_threshold if "v_threshold" in dir() else None,
    "plateau_slope": plateau_slope_pct if "plateau_slope_pct" in dir() else None,
    "v_recommend": v_recommend if "v_recommend" in dir() else None,
    "tau_us": tau_bc if "tau_bc" in dir() else None,
    "true_rate_cps": lam * 1e6 if "lam" in dir() else None,
    "bg_rate_cps": background_rate if "background_rate" in dir() else None,
    "fano": fano if "fano" in dir() else None,
    "distance_A": coeffs[0] if "coeffs" in dir() else None,
}
# Replace existing entry for same tube or append
tube_summary = [e for e in tube_summary if e["tube_id"] != TUBE_ID]
tube_summary.append(entry)

print(f"Stored results for {TUBE_ID}. Total tubes: {len(tube_summary)}")

In [ ]:
# ── Comparison table ───────────────────────────────────────────────────────
if not tube_summary:
    print("No results yet. Run the accumulate cell after measuring each tube.")
else:
    import pandas as pd

    df = pd.DataFrame(tube_summary).set_index("tube_id")
    df.columns = [
        "Size",
        "V_thresh (V)",
        "Slope (%/100V)",
        "V_op (V)",
        "τ (µs)",
        "True rate (cps)",
        "BG (cps)",
        "Fano",
        "A (cps·cm²)",
    ]
    display(
        df.style.format(
            {
                "V_thresh (V)": "{:.0f}",
                "Slope (%/100V)": "{:.2f}",
                "V_op (V)": "{:.0f}",
                "τ (µs)": "{:.1f}",
                "True rate (cps)": "{:.1f}",
                "BG (cps)": "{:.3f}",
                "Fano": "{:.3f}",
                "A (cps·cm²)": "{:.3g}",
            },
            na_rep="—",
        )
    )

In [ ]:
# ── Side-by-side plateau comparison ───────────────────────────────────────
# Overlay plateau curves from multiple tubes if CSV exports were saved.
# Format: run section 1, save plateau_voltages / plateau_rates to a dict.

if "all_plateaus" not in dir():
    all_plateaus = {}

if "plateau_voltages" in dir():
    all_plateaus[TUBE_ID] = (np.array(plateau_voltages), np.array(plateau_rates))

if len(all_plateaus) >= 2:
    fig, ax = plt.subplots(figsize=(10, 5))
    for tid, (vs, rs) in all_plateaus.items():
        ax.plot(vs, rs / rs.max(), "o-", ms=4, label=tid)
    ax.set_xlabel("High Voltage (V)")
    ax.set_ylabel("Normalised count rate")
    ax.set_title("Plateau comparison (normalised)")
    ax.legend()
    plt.tight_layout()
    plt.savefig("plateau_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print(
        f"Plateaus stored: {list(all_plateaus.keys())}  (need ≥ 2 for comparison plot)"
    )

---

## 7 · Summary JSON

Raw CSVs were already saved inline after each measurement. This cell writes a
`summary.json` consolidating all key derived results into a single file.


In [ ]:
import json

ts = datetime.now()
OUT_DIR.mkdir(parents=True, exist_ok=True)
summary_path = OUT_DIR / f"{ts:%Y-%m-%d}_{ts:%H%M}_summary_{TUBE_ID}_{SOURCE_ID}.json"


def _get(name):
    return globals().get(name)


summary = {
    "tube_id": TUBE_ID,
    "tube_size": TUBE_SIZE,
    "source_id": SOURCE_ID,
    "timestamp": ts.isoformat(),
    "operating_voltage_V": V_OPERATE,
    "plateau": {
        "threshold_V": (
            int(_get("v_threshold")) if _get("v_threshold") is not None else None
        ),
        "slope_pct_100V": (
            round(float(_get("plateau_slope_pct")), 3)
            if _get("plateau_slope_pct") is not None
            and not math.isnan(_get("plateau_slope_pct"))
            else None
        ),
        "recommended_V": (
            int(_get("v_recommend")) if _get("v_recommend") is not None else None
        ),
    },
    "dead_time": {
        "tau_mle_us": (
            round(float(_get("tau_mle")), 2) if _get("tau_mle") is not None else None
        ),
        "tau_bc_us": (
            round(float(_get("tau_bc")), 2) if _get("tau_bc") is not None else None
        ),
        "tau_hist_us": (
            round(float(_get("tau_hist")), 2) if _get("tau_hist") is not None else None
        ),
        "chi2_reduced": (
            round(float(_get("chi2r")), 3) if _get("chi2r") is not None else None
        ),
        "true_rate_cps": (
            round(float(_get("lam") * 1e6), 2) if _get("lam") is not None else None
        ),
    },
    "background_cps": (
        round(float(_get("background_rate")), 4)
        if _get("background_rate") is not None
        else None
    ),
    "poisson": {
        "n_samples": (
            len(_get("poisson_counts")) if _get("poisson_counts") is not None else None
        ),
        "fano": round(float(_get("fano")), 4) if _get("fano") is not None else None,
        "chi2_p": (
            round(float(_get("p_value")), 4)
            if _get("p_value") is not None and not math.isnan(_get("p_value"))
            else None
        ),
    },
}

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"Saved: {summary_path}")
print()
print(json.dumps(summary, indent=2))

---

## 8 · Cleanup


In [ ]:
# Safe shutdown: stop counting, restore defaults, close port
try:
    gm.set_counting(False)
    gm.set_stream(0)
    gm.set_speaker(gm=False, ready=False)
    gm.set_voltage(V_OPERATE)
    gm.close()
    print("Device closed cleanly.")
except Exception as exc:
    print(f"Cleanup warning: {exc}")